# Tutorial 2: Parameter Sweep for Threshold Optimization

This tutorial demonstrates how to use `boldgenotyper-sweep` to find the optimal clustering threshold for your dataset.

## Learning Objectives

By the end of this tutorial, you will:
- Understand why threshold optimization is important
- Learn to run a parameter sweep analysis
- Interpret elbow plots and stability metrics
- Make data-driven decisions about clustering thresholds
- Use sweep results to improve your final analysis

## Prerequisites

- Completed Tutorial 1 (Basic Genotyping Workflow)
- Understanding of hierarchical clustering concepts
- Example dataset: `Sphyrna_lewini_scallopedhammerhead.tsv`

## Why Optimize Thresholds?

The clustering threshold determines how sequences are grouped into genotypes:
- **Too low**: Over-splitting, many singleton genotypes, loss of biological signal
- **Too high**: Under-splitting, lumping distinct genotypes together
- **Optimal**: Captures biological structure, maximizes assignment efficiency

Parameter sweep helps find the "elbow point" where adding more genotypes provides diminishing returns.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

## Step 1: Run Parameter Sweep

The parameter sweep will test multiple thresholds and generate comparison metrics.

### Default thresholds tested:
- 0.005 (0.5%)
- 0.010 (1.0%)
- 0.015 (1.5%)
- 0.020 (2.0%)
- 0.025 (2.5%)
- 0.030 (3.0%)

You can customize this range based on your organism's expected COI variation.

In [ ]:
# Run parameter sweep with default thresholds
!boldgenotyper-sweep ../data/Sphyrna_lewini_scallopedhammerhead.tsv \
  --output ../data/Sphyrna_lewini_sweep/ \
  --threads 4

In [ ]:
# Alternative: Run with custom threshold range
# Uncomment and modify if you need a different range

# !boldgenotyper-sweep ../data/Sphyrna_lewini_scallopedhammerhead.tsv \
#   --thresholds 0.01,0.015,0.02,0.025,0.03,0.04 \
#   --output ../data/Sphyrna_lewini_sweep/ \
#   --threads 4

## Step 2: Examine Output Files

The parameter sweep generates several output files for analysis.

In [ ]:
# List output files
!ls -lh ../data/Sphyrna_lewini_sweep/

### Key output files:
1. **sweep_summary.csv** - Metrics for each threshold
2. **threshold_stability.pdf** - Multi-panel visualization
3. **elbow_plot.pdf** - Elbow point detection
4. **group_membership_tracking.csv** - Clustering stability
5. **recommendations.txt** - Automated interpretation
6. **runs/** - Individual analysis outputs for each threshold

## Step 3: Load and Analyze Summary Metrics

In [ ]:
# Load sweep summary
summary = pd.read_csv("../data/Sphyrna_lewini_sweep/sweep_summary.csv")

print("Parameter Sweep Summary")
print("="*80)
print(summary.to_string(index=False))
print("="*80)

In [ ]:
# Visualize key metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Number of genotypes vs threshold
axes[0, 0].plot(summary['threshold'], summary['n_genotypes'], 'o-', linewidth=2, markersize=8)
axes[0, 0].set_xlabel('Clustering Threshold')
axes[0, 0].set_ylabel('Number of Genotypes')
axes[0, 0].set_title('Genotype Count by Threshold')
axes[0, 0].grid(True, alpha=0.3)

# 2. Assignment efficiency vs threshold
if 'pct_assigned' in summary.columns:
    axes[0, 1].plot(summary['threshold'], summary['pct_assigned'], 'o-', 
                    linewidth=2, markersize=8, color='green')
    axes[0, 1].set_xlabel('Clustering Threshold')
    axes[0, 1].set_ylabel('Assignment Efficiency (%)')
    axes[0, 1].set_title('Assignment Efficiency by Threshold')
    axes[0, 1].grid(True, alpha=0.3)

# 3. Singleton percentage vs threshold
if 'pct_singletons' in summary.columns:
    axes[1, 0].plot(summary['threshold'], summary['pct_singletons'], 'o-', 
                    linewidth=2, markersize=8, color='orange')
    axes[1, 0].set_xlabel('Clustering Threshold')
    axes[1, 0].set_ylabel('Singleton Genotypes (%)')
    axes[1, 0].set_title('Singleton Percentage by Threshold')
    axes[1, 0].grid(True, alpha=0.3)

# 4. Clustering stability
if 'stability_score' in summary.columns:
    axes[1, 1].plot(summary['threshold'], summary['stability_score'], 'o-', 
                    linewidth=2, markersize=8, color='purple')
    axes[1, 1].set_xlabel('Clustering Threshold')
    axes[1, 1].set_ylabel('Stability Score (Jaccard)')
    axes[1, 1].set_title('Clustering Stability by Threshold')
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 4: Detect Elbow Point

The elbow point is where the curve begins to flatten, indicating diminishing returns.

In [ ]:
# Calculate second derivative to find elbow
from scipy.ndimage import gaussian_filter1d

# Smooth the curve
thresholds = summary['threshold'].values
n_genotypes = summary['n_genotypes'].values

# Calculate first and second derivatives
first_deriv = np.gradient(n_genotypes, thresholds)
second_deriv = np.gradient(first_deriv, thresholds)

# Find maximum of second derivative (sharpest curve change)
elbow_idx = np.argmax(np.abs(second_deriv))
elbow_threshold = thresholds[elbow_idx]

print(f"Detected elbow point: {elbow_threshold:.3f}")
print(f"Genotypes at elbow: {n_genotypes[elbow_idx]:.0f}")

In [ ]:
# Visualize elbow detection
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Main curve with elbow point
axes[0].plot(thresholds, n_genotypes, 'o-', linewidth=2, markersize=8, label='Observed')
axes[0].axvline(elbow_threshold, color='red', linestyle='--', linewidth=2, 
                label=f'Elbow: {elbow_threshold:.3f}')
axes[0].scatter([elbow_threshold], [n_genotypes[elbow_idx]], 
                color='red', s=200, zorder=5, marker='*')
axes[0].set_xlabel('Clustering Threshold')
axes[0].set_ylabel('Number of Genotypes')
axes[0].set_title('Elbow Point Detection')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Second derivative
axes[1].plot(thresholds, second_deriv, 'o-', linewidth=2, markersize=8, color='purple')
axes[1].axvline(elbow_threshold, color='red', linestyle='--', linewidth=2)
axes[1].axhline(0, color='black', linestyle='-', linewidth=0.5)
axes[1].set_xlabel('Clustering Threshold')
axes[1].set_ylabel('Second Derivative')
axes[1].set_title('Curvature Analysis')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5: Analyze Clustering Stability

Stability measures how consistent genotype assignments are between consecutive thresholds.

In [ ]:
# Load group membership tracking
tracking_file = "../data/Sphyrna_lewini_sweep/group_membership_tracking.csv"
if Path(tracking_file).exists():
    tracking = pd.read_csv(tracking_file)
    print("Clustering stability across thresholds:")
    print(tracking.head(20))
else:
    print("Group membership tracking file not found.")

## Step 6: Read Automated Recommendations

In [ ]:
# Load recommendations
rec_file = "../data/Sphyrna_lewini_sweep/recommendations.txt"
if Path(rec_file).exists():
    with open(rec_file, 'r') as f:
        recommendations = f.read()
    print("="*80)
    print("AUTOMATED RECOMMENDATIONS")
    print("="*80)
    print(recommendations)
    print("="*80)
else:
    print("Recommendations file not found.")

## Step 7: Make Threshold Decision

Consider multiple factors when choosing your threshold.

In [ ]:
# Decision framework
print("THRESHOLD DECISION FRAMEWORK")
print("="*80)
print("\n1. ELBOW POINT:")
print(f"   Detected elbow: {elbow_threshold:.3f}")
print(f"   Interpretation: Optimal balance of genotype resolution")

if 'pct_assigned' in summary.columns:
    # Find threshold with maximum assignment efficiency
    max_eff_idx = summary['pct_assigned'].idxmax()
    max_eff_threshold = summary.loc[max_eff_idx, 'threshold']
    print(f"\n2. MAXIMUM ASSIGNMENT EFFICIENCY:")
    print(f"   Threshold: {max_eff_threshold:.3f}")
    print(f"   Efficiency: {summary.loc[max_eff_idx, 'pct_assigned']:.1f}%")
    print(f"   Interpretation: Best sample-to-genotype matching")

if 'pct_singletons' in summary.columns:
    # Find threshold with <20% singletons
    acceptable = summary[summary['pct_singletons'] < 20]
    if len(acceptable) > 0:
        rec_threshold = acceptable.iloc[0]['threshold']
        print(f"\n3. CONSERVATIVE APPROACH (<20% singletons):")
        print(f"   Threshold: {rec_threshold:.3f}")
        print(f"   Interpretation: Avoids over-splitting")

print(f"\n4. BIOLOGICAL VALIDATION:")
print(f"   - Do major genotypes correspond to known populations?")
print(f"   - Are geographic patterns biologically plausible?")
print(f"   - Does threshold align with literature for this taxon?")

print("\n" + "="*80)
print(f"RECOMMENDED THRESHOLD: {elbow_threshold:.3f}")
print("="*80)

## Step 8: Run Final Analysis with Optimized Threshold

Use the recommended threshold for your final analysis.

In [ ]:
# Run final analysis with optimized threshold
optimal_threshold = elbow_threshold

!boldgenotyper ../data/Sphyrna_lewini_scallopedhammerhead.tsv \
  --clustering-threshold {optimal_threshold} \
  --build-tree \
  --output ../data/Sphyrna_lewini_optimized/ \
  --threads 4

## Step 9: Compare Default vs Optimized Results

In [ ]:
# Load both result sets
default_results = pd.read_csv("../data/Sphyrna_lewini_tutorial/Sphyrna_lewini_annotated.csv")
optimized_results = pd.read_csv("../data/Sphyrna_lewini_optimized/Sphyrna_lewini_annotated.csv")

print("COMPARISON: Default (0.03) vs Optimized Threshold")
print("="*80)
print(f"\nDefault threshold: 0.030")
print(f"  Genotypes: {default_results['genotype'].nunique()}")
print(f"  Samples: {len(default_results)}")

print(f"\nOptimized threshold: {optimal_threshold:.3f}")
print(f"  Genotypes: {optimized_results['genotype'].nunique()}")
print(f"  Samples: {len(optimized_results)}")

print(f"\nDifference:")
genotype_diff = optimized_results['genotype'].nunique() - default_results['genotype'].nunique()
print(f"  Genotype count: {genotype_diff:+d}")
print(f"  Interpretation: {'More fine-scale resolution' if genotype_diff > 0 else 'More conservative clustering'}")
print("="*80)

## Step 10: Document Methods for Publication

Generate publication-ready methods text.

In [ ]:
# Generate methods text
methods_text = f"""
METHODS - Clustering Threshold Optimization

To determine the optimal clustering threshold for COI genotype assignment, we 
performed a parameter sweep analysis using boldgenotyper-sweep (v0.1.0). We 
tested thresholds ranging from {summary['threshold'].min():.3f} to 
{summary['threshold'].max():.3f} and evaluated genotype count, assignment 
efficiency, and clustering stability for each threshold.

The optimal threshold of {optimal_threshold:.3f} was selected based on elbow 
point detection, which identifies the threshold where increasing stringency 
provides diminishing returns in genotype resolution. This threshold yielded 
{optimized_results['genotype'].nunique()} genotypes from 
{len(optimized_results)} samples, balancing fine-scale genetic resolution with 
robust genotype assignments.
"""

print(methods_text)

# Save to file
with open("../data/Sphyrna_lewini_sweep/methods_text.txt", 'w') as f:
    f.write(methods_text)
    
print("\nMethods text saved to: ../data/Sphyrna_lewini_sweep/methods_text.txt")

## Key Takeaways

1. **Data-Driven Decisions**: Parameter sweep replaces arbitrary threshold choices
2. **Elbow Method**: Identifies optimal balance between resolution and stability
3. **Multiple Criteria**: Consider elbow point, assignment efficiency, and biology
4. **Reproducibility**: Automated recommendations provide objective justification
5. **Publication-Ready**: Generate methods text directly from sweep results

## Best Practices

1. **Always run parameter sweep** for new datasets or taxonomic groups
2. **Test appropriate range** based on expected COI variation:
   - Fish/vertebrates: 0.005-0.03
   - Invertebrates: 0.01-0.05
   - Within-species: 0.001-0.01
3. **Validate biologically** by checking if genotypes match known populations
4. **Document thoroughly** in methods section

## Next Steps

- **Tutorial 3**: Quality control using comparative analysis
- **Tutorial 4**: Custom shapefiles for non-marine organisms
- **Tutorial 5**: Export for population genetics analysis

## Additional Resources

- Parameter Sweep Guide: `../PARAMETER_SWEEP_GUIDE.md`
- Interpretation strategies and troubleshooting
- Advanced usage examples